# 📊 Preprocessing Pipeline

**Mục tiêu:**
1. Load raw data
2. **FIX DATA LEAKAGE** trong num_post, num_real, num_fake, post_ratio
3. Clean texts cho TF-IDF và BERT
4. Tạo features
5. Remove leaked columns và user identifiers
6. Save processed data

In [41]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# Add project root to path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\Vietnamese-Fake-News-Detection


In [42]:
# Import preprocessing functions
from src.s1_preprocessing.n0_feature_extraction_function import create_all_features
from src.s1_preprocessing.n0_text_cleaning_function import clean_for_tfidf, clean_for_bert

## 1. Configuration

In [43]:
# Paths
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PREPROCESSED_DIR = PROJECT_ROOT / "data" / "preprocessed"

# Input/Output
INPUT_FILE = DATA_RAW_DIR / "data.csv"
OUTPUT_FILE = DATA_PREPROCESSED_DIR / "preprocessed_data.csv"

# Create output directory
DATA_PREPROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")

Input: D:\Vietnamese-Fake-News-Detection\data\raw\data.csv
Output: D:\Vietnamese-Fake-News-Detection\data\preprocessed\preprocessed_data.csv


## 2. Load Raw Data

In [44]:
# Load data
df_raw = pd.read_csv(INPUT_FILE)
print(f"Loaded {len(df_raw)} rows, {len(df_raw.columns)} columns")
print(f"\nColumns: {df_raw.columns.tolist()}")

Loaded 4736 rows, 26 columns

Columns: ['id', 'user_name', 'timestamp_post', 'post_message', 'label', 'num_char', 'num_emoji', 'num_url', 'num_hashtag', 'num_post', 'num_real', 'num_fake', 'post_ratio', 'num_like', 'num_cmt', 'num_share', 'pixel', 'image', 'num_image', 'hour', 'weekday', 'day', 'month', 'year', 'time', 'min']


In [45]:
# Quick overview
print("Label distribution:")
print(df_raw['label'].value_counts())
print(f"\nUnique users: {df_raw['user_name'].nunique()}")

Label distribution:
label
0    3929
1     807
Name: count, dtype: int64

Unique users: 3446


## 3. ⚠️ FIX DATA LEAKAGE

**Vấn đề:** 
- `num_post`, `num_real`, `num_fake` được tính từ TOÀN BỘ dataset → DATA LEAKAGE
- `post_ratio` cũng bị leak

**Giải pháp:**
- Tính lại dựa trên HISTORICAL data (các bài trước đó của user)
- Post đầu tiên của user → num_post=0, num_real=0, num_fake=0

In [46]:
def fix_user_historical_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Fix data leakage bằng cách tính lại num_post, num_real, num_fake
    dựa trên historical data (các bài trước đó của user)
    
    Logic:
    - Sort posts của mỗi user theo timestamp (tie-breaker: id)
    - Với mỗi post, chỉ tính từ các posts TRƯỚC đó
    - Post đầu tiên của user → num_post=0, num_real=0, num_fake=0
    """
    df = df.copy()
    
    # Backup original values để so sánh
    df['_orig_num_post'] = df['num_post']
    df['_orig_num_real'] = df['num_real']
    df['_orig_num_fake'] = df['num_fake']
    if 'post_ratio' in df.columns:
        df['_orig_post_ratio'] = df['post_ratio']
    
    # Initialize new columns
    df['num_post'] = 0
    df['num_real'] = 0
    df['num_fake'] = 0
    
    # Sort by user, then timestamp, then id (tie-breaker)
    df = df.sort_values(['user_name', 'timestamp_post', 'id']).reset_index(drop=True)
    
    # Calculate historical counts for each user
    print("Calculating historical features for each user...")
    
    users = df['user_name'].unique()
    
    for user in tqdm(users, desc="Processing users"):
        user_mask = df['user_name'] == user
        user_indices = df[user_mask].index.tolist()
        
        cumsum_real = 0
        cumsum_fake = 0
        
        for i, idx in enumerate(user_indices):
            df.loc[idx, 'num_post'] = i  # Number of posts BEFORE this one
            df.loc[idx, 'num_real'] = cumsum_real
            df.loc[idx, 'num_fake'] = cumsum_fake
            
            current_label = df.loc[idx, 'label']
            if current_label == 0:  # Real
                cumsum_real += 1
            else:  # Fake
                cumsum_fake += 1
    
    return df

In [47]:
# Fix data leakage
df = fix_user_historical_features(df_raw)

Calculating historical features for each user...


Processing users: 100%|██████████| 3446/3446 [00:02<00:00, 1560.85it/s]


### 3.1 Verification: Single Post Users

In [48]:
print("="*60)
print("VERIFICATION: SINGLE POST USERS")
print("="*60)

single_post_mask = df.groupby('user_name')['user_name'].transform('count') == 1
single_post_df = df[single_post_mask]
print(f"\nUsers with single post: {len(single_post_df)}")
print(f"  - All num_post=0: {(single_post_df['num_post'] == 0).all()}")
print(f"  - All num_real=0: {(single_post_df['num_real'] == 0).all()}")
print(f"  - All num_fake=0: {(single_post_df['num_fake'] == 0).all()}")

VERIFICATION: SINGLE POST USERS

Users with single post: 3240
  - All num_post=0: True
  - All num_real=0: True
  - All num_fake=0: True


### 3.2 Examples: Multi-Post Users

In [49]:
def show_user_posts(df, num_posts, max_display=None):
    """Hiển thị tất cả posts của user có đúng num_posts bài đăng"""
    user_counts = df.groupby('user_name').size()
    users_with_n = user_counts[user_counts == num_posts].index.tolist()
    
    if not users_with_n:
        print(f"Không có user nào với {num_posts} posts")
        return
    
    sample_user = users_with_n[0]
    cols = ['timestamp_post', 'label', 'num_post', 'num_real', 'num_fake', 
            '_orig_num_post', '_orig_num_real', '_orig_num_fake']
    
    user_df = df[df['user_name'] == sample_user][cols]
    if max_display:
        user_df = user_df.head(max_display)
    
    print(f"\n{'='*80}")
    print(f"USER VỚI {num_posts} POSTS (User: {sample_user[:30]}...)")
    print(f"{'='*80}")
    print(user_df.to_string())
    print(f"\n✓ First post: num_post={user_df.iloc[0]['num_post']}, num_real={user_df.iloc[0]['num_real']}, num_fake={user_df.iloc[0]['num_fake']}")
    if len(user_df) > 1:
        print(f"✓ Last post: num_post={user_df.iloc[-1]['num_post']}, num_real={user_df.iloc[-1]['num_real']}, num_fake={user_df.iloc[-1]['num_fake']}")

In [50]:
# User với 2 posts
show_user_posts(df, 2)


USER VỚI 2 POSTS (User: -1.6396E+18...)
    timestamp_post  label  num_post  num_real  num_fake  _orig_num_post  _orig_num_real  _orig_num_fake
10      1589606321      0         0         0         0               2               2               0
11      1590021454      0         1         1         0               2               2               0

✓ First post: num_post=0, num_real=0, num_fake=0
✓ Last post: num_post=1, num_real=1, num_fake=0


In [51]:
# User với 3 posts
show_user_posts(df, 3)


USER VỚI 3 POSTS (User: -6.61911E+18...)
     timestamp_post  label  num_post  num_real  num_fake  _orig_num_post  _orig_num_real  _orig_num_fake
139      1589983353      0         0         0         0               3               3               0
140      1590203515      0         1         1         0               3               3               0
141      1592280286      0         2         2         0               3               3               0

✓ First post: num_post=0, num_real=0, num_fake=0
✓ Last post: num_post=2, num_real=2, num_fake=0


In [52]:
# User với 5 posts
show_user_posts(df, 5)


USER VỚI 5 POSTS (User: -2.85614E+18...)
    timestamp_post  label  num_post  num_real  num_fake  _orig_num_post  _orig_num_real  _orig_num_fake
30      1588983037      0         0         0         0               5               5               0
31      1589701948      0         1         1         0               5               5               0
32      1590497280      0         2         2         0               5               5               0
33      1591227217      0         3         3         0               5               5               0
34      1591745374      0         4         4         0               5               5               0

✓ First post: num_post=0, num_real=0, num_fake=0
✓ Last post: num_post=4, num_real=4, num_fake=0


In [53]:
# User với nhiều posts nhất
user_counts = df.groupby('user_name').size()
max_posts = user_counts.max()
print(f"\nUser có nhiều posts nhất: {max_posts} posts")
show_user_posts(df, max_posts)


User có nhiều posts nhất: 90 posts

USER VỚI 90 POSTS (User: 383b4190873d1aa2f9b2124a286ef1...)
      timestamp_post  label  num_post  num_real  num_fake  _orig_num_post  _orig_num_real  _orig_num_fake
1154      1583224136      0         0         0         0              90              90               0
1155      1583359015      0         1         1         0              90              90               0
1156      1583454778      0         2         2         0              90              90               0
1157      1583465342      0         3         3         0              90              90               0
1158      1583471075      0         4         4         0              90              90               0
1159      1583524584      0         5         5         0              90              90               0
1160      1583538800      0         6         6         0              90              90               0
1161      1583541794      0         7         7        

### 3.3 Final Verification

In [54]:
print("="*60)
print("FINAL VERIFICATION")
print("="*60)

# num_real + num_fake should equal num_post
verify = (df['num_real'] + df['num_fake'] == df['num_post']).all()
print(f"\n1. num_real + num_fake = num_post: {verify}")

# First post of each user should have all zeros
first_posts = df.groupby('user_name').first()
first_post_check = (
    (first_posts['num_post'] == 0).all() and
    (first_posts['num_real'] == 0).all() and
    (first_posts['num_fake'] == 0).all()
)
print(f"2. First post of each user has num_post=num_real=num_fake=0: {first_post_check}")

# Compare with original
changed = (df['num_post'] != df['_orig_num_post']).sum()
print(f"3. Records changed: {changed} / {len(df)} ({changed/len(df)*100:.1f}%)")

FINAL VERIFICATION

1. num_real + num_fake = num_post: True
2. First post of each user has num_post=num_real=num_fake=0: True
3. Records changed: 4736 / 4736 (100.0%)


### 3.4 Remove Leaked Columns

In [55]:
# Columns to remove (leaked data)
leaked_cols = ['_orig_num_post', '_orig_num_real', '_orig_num_fake', 'post_ratio']
if '_orig_post_ratio' in df.columns:
    leaked_cols.append('_orig_post_ratio')

existing_leaked = [c for c in leaked_cols if c in df.columns]
print(f"Removing leaked columns: {existing_leaked}")
df = df.drop(columns=existing_leaked, errors='ignore')
print(f"✓ Removed {len(existing_leaked)} leaked columns")

Removing leaked columns: ['_orig_num_post', '_orig_num_real', '_orig_num_fake', 'post_ratio', '_orig_post_ratio']
✓ Removed 5 leaked columns


## 4. Text Cleaning

In [56]:
print("Cleaning texts for TF-IDF...")
tqdm.pandas(desc="TF-IDF")
df['text_tfidf'] = df['post_message'].progress_apply(
    lambda x: clean_for_tfidf(x) if pd.notna(x) else ""
)

print("\nCleaning texts for BERT...")
tqdm.pandas(desc="BERT")
df['text_bert'] = df['post_message'].progress_apply(
    lambda x: clean_for_bert(x) if pd.notna(x) else ""
)

print("\n✓ Created text_tfidf and text_bert columns")

Cleaning texts for TF-IDF...


TF-IDF: 100%|██████████| 4736/4736 [00:00<00:00, 8522.75it/s]



Cleaning texts for BERT...


BERT: 100%|██████████| 4736/4736 [00:02<00:00, 2115.18it/s]


✓ Created text_tfidf and text_bert columns


In [57]:
# Sample cleaned text
print("Sample cleaned text:")
long_texts = df[df['post_message'].str.len() > 100]
if len(long_texts) > 0:
    sample_idx = long_texts.index[0]
    print(f"\nOriginal ({len(df.loc[sample_idx, 'post_message'])} chars):")
    print(df.loc[sample_idx, 'post_message'][:200], "...")
    print(f"\nTF-IDF ({len(df.loc[sample_idx, 'text_tfidf'])} chars):")
    print(df.loc[sample_idx, 'text_tfidf'][:200], "...")

Sample cleaned text:

Original (121 chars):
Bộ Y_tế chiều 7/5 công_bố thêm 17 trường_hợp mắc Covid-19 đều là những ca mới nhập_cảnh về Việt_Nam , được cách_ly ngay . ...

TF-IDF (117 chars):
bộ y_tế chiều 7 5 công_bố thêm 17 trường_hợp mắc covid 19 đều là những ca mới nhập_cảnh về việt_nam được cách_ly ngay ...


## 5. Create Features

In [58]:
# Create all features
df = create_all_features(df)

Creating text features...
Creating engagement features...
Creating user features...
Creating time features...

Created 40 new features:
  - feat_num_chars
  - feat_num_words
  - feat_num_sentences
  - feat_num_exclamation
  - feat_num_question
  - feat_num_uppercase_words
  - feat_uppercase_ratio
  - feat_num_emojis
  - feat_num_urls
  - feat_num_hashtags
  - feat_num_mentions
  - feat_num_digits
  - feat_digit_ratio
  - feat_num_special_chars
  - feat_avg_word_length
  - feat_num_long_words
  - feat_num_short_words
  - feat_num_quoted
  - feat_has_all_caps
  - feat_num_repeated_chars
  - feat_num_newlines
  - feat_punctuation_ratio
  - feat_engagement_total
  - feat_like_ratio
  - feat_comment_ratio
  - feat_like_per_char
  - feat_cmt_per_char
  - feat_share_per_char
  - feat_like_cmt_ratio
  - feat_share_like_ratio
  - feat_fake_ratio
  - feat_is_weekend
  - feat_is_night
  - feat_is_morning
  - feat_is_afternoon
  - feat_is_evening
  - feat_hour_sin
  - feat_hour_cos
  - feat_day_si

In [59]:
# List all features
feature_cols = [col for col in df.columns if col.startswith('feat_')]
print(f"\nTotal features created: {len(feature_cols)}")
for col in feature_cols:
    print(f"  - {col}")


Total features created: 40
  - feat_num_chars
  - feat_num_words
  - feat_num_sentences
  - feat_num_exclamation
  - feat_num_question
  - feat_num_uppercase_words
  - feat_uppercase_ratio
  - feat_num_emojis
  - feat_num_urls
  - feat_num_hashtags
  - feat_num_mentions
  - feat_num_digits
  - feat_digit_ratio
  - feat_num_special_chars
  - feat_avg_word_length
  - feat_num_long_words
  - feat_num_short_words
  - feat_num_quoted
  - feat_has_all_caps
  - feat_num_repeated_chars
  - feat_num_newlines
  - feat_punctuation_ratio
  - feat_engagement_total
  - feat_like_ratio
  - feat_comment_ratio
  - feat_like_per_char
  - feat_cmt_per_char
  - feat_share_per_char
  - feat_like_cmt_ratio
  - feat_share_like_ratio
  - feat_fake_ratio
  - feat_is_weekend
  - feat_is_night
  - feat_is_morning
  - feat_is_afternoon
  - feat_is_evening
  - feat_hour_sin
  - feat_hour_cos
  - feat_day_sin
  - feat_day_cos


## 6. Remove User Identifiers

In [60]:
# Remove user_name to prevent identity leakage
cols_to_remove = ['user_name']
print(f"Removing columns: {cols_to_remove}")
df = df.drop(columns=[c for c in cols_to_remove if c in df.columns], errors='ignore')
print(f"\nFinal columns: {len(df.columns)}")

Removing columns: ['user_name']

Final columns: 66


## 7. Save Preprocessed Data

In [61]:
# Save
df.to_csv(OUTPUT_FILE, index=False)
print(f"✓ Saved to: {OUTPUT_FILE}")
print(f"  Rows: {len(df)}")
print(f"  Columns: {len(df.columns)}")

✓ Saved to: D:\Vietnamese-Fake-News-Detection\data\preprocessed\preprocessed_data.csv
  Rows: 4736
  Columns: 66


## 8. Summary

In [62]:
print("="*60)
print("PREPROCESSING COMPLETE")
print("="*60)

print(f"\n📊 Dataset shape: {df.shape}")
print(f"\n📝 Label distribution:")
print(df['label'].value_counts())

print(f"\n🔧 Feature columns ({len(feature_cols)}):")
for col in feature_cols:
    print(f"  - {col}")

print(f"\n📄 Text columns: text_tfidf, text_bert")

print(f"\n✅ Data leakage FIXED:")
print(f"   - num_post, num_real, num_fake now use historical data only")
print(f"   - post_ratio removed")
print(f"   - user_name removed to prevent identity leakage")

print(f"\n💾 Output saved to: {OUTPUT_FILE}")

PREPROCESSING COMPLETE

📊 Dataset shape: (4736, 66)

📝 Label distribution:
label
0    3929
1     807
Name: count, dtype: int64

🔧 Feature columns (40):
  - feat_num_chars
  - feat_num_words
  - feat_num_sentences
  - feat_num_exclamation
  - feat_num_question
  - feat_num_uppercase_words
  - feat_uppercase_ratio
  - feat_num_emojis
  - feat_num_urls
  - feat_num_hashtags
  - feat_num_mentions
  - feat_num_digits
  - feat_digit_ratio
  - feat_num_special_chars
  - feat_avg_word_length
  - feat_num_long_words
  - feat_num_short_words
  - feat_num_quoted
  - feat_has_all_caps
  - feat_num_repeated_chars
  - feat_num_newlines
  - feat_punctuation_ratio
  - feat_engagement_total
  - feat_like_ratio
  - feat_comment_ratio
  - feat_like_per_char
  - feat_cmt_per_char
  - feat_share_per_char
  - feat_like_cmt_ratio
  - feat_share_like_ratio
  - feat_fake_ratio
  - feat_is_weekend
  - feat_is_night
  - feat_is_morning
  - feat_is_afternoon
  - feat_is_evening
  - feat_hour_sin
  - feat_hour_cos